# Incremental Embedding Updates at Scale (Part 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qdrant/examples/blob/master/temporal-data-drift/sync_at_scale_with_digests.ipynb)

[Part 1](https://qdrant.tech/documentation/tutorials-operations/incremental-embedding-updates/) keeps a Qdrant collection in sync with changing documentation by reconciling the full list of current chunks against the whole Qdrant collection on every run. It is a simple setup that works, but since every run reads the whole collection, the sync cost grows linearly with the size of the corpus.

This notebook proposes an alternative, more involved approach for the case where scale is a real factor (millions or billions of chunks). On top of the Part 1 idea it adds a small summary layer, so each run scans a fixed-size summary and touches only the parts of the collection that changed.

## The Idea

The method splits the chunks into a fixed number of **buckets** and gives each bucket a **digest**: a single number that summarizes everything in it. A bucket's digest is built from a small per-chunk value, the chunk's **contribution**, which depends on the chunk's position and its text. So if any chunk's text or position changes, a new chunk lands in the bucket, or one leaves it, the bucket's digest changes and we know that bucket needs a look. If the digest is unchanged, the bucket's contents are unchanged and we skip it.

We keep two collections:

- the **chunks collection**, whose embeddings serve vector search: the Part 1 collection plus one extra payload field, `sync_bucket`. The point ID is unchanged from Part 1 (a stable ID derived from the chunk's address). We do not store the per-chunk contributions; they are cheap to recompute, and only the per-bucket digest is kept.
- a small **summary collection**, holding each bucket and its digest. This is the sync state. For a given bucket count it is a fixed size, and it stays a small fraction of the collection even at large scale.

Two things make it work:

- **Comparison instead of scanning.** We compute the digests from the current source and compare them against the digests Qdrant already stored. Only the buckets whose digest differs get reconciled; the rest of the collection is never read.
- **XOR to combine contributions into a bucket digest.** XOR because:
  1. it is order-independent, so however the chunks come back, the same set of chunks gives the same digest;
  2. it folds any number of contributions into one fixed-width number with one cheap operation, so the summary stays small no matter how big the bucket;
  3. it is reversible (XORing a value twice cancels it), so at scale a bucket's digest could be patched for a single added or removed chunk instead of rebuilt. This notebook recomputes each changed bucket from scratch for clarity, but that reversibility is why XOR is the natural choice.

### The Math

Say we use 2^16 = 65536 buckets. For a corpus of 1,000,000 chunks that is about 15 chunks per bucket. Each run compares the 65536 digests to find the changed buckets, then reads chunks only from those buckets. If 50 chunks changed, they sit in at most ~50 buckets, so we read on the order of 50 x 15 = 750 chunks, not 1,000,000. Part 1 reads all 1,000,000 every run.

We also pack the 65536 digests into a handful of points rather than one point per bucket, so the whole summary is a single read. The build section below explains why and how.

To keep the summary printable, this notebook uses **16 buckets**. All the code is written against one constant, `N_BUCKETS`; set it to 65536 for production, and higher for very large corpora so each bucket stays small.

## Prerequisites

This notebook uses Qdrant Cloud and its Free Tier Inference. Create a Free Tier Qdrant Cloud cluster and paste its URL and API key into the client cell below.

In [ ]:
%pip install -q "qdrant-client>=1.18"

In [ ]:
from qdrant_client import QdrantClient, models

# Replace url and api_key with your own from https://cloud.qdrant.io
client = QdrantClient(
    url="https://xyz-example.qdrant.io:6333",
    api_key="<your-api-key>",
    cloud_inference=True
)

## The Chunks

We use the same documentation chunks as Part 1. Each chunk has an address (which page, which section, which piece of that section) and its text. Two derived values carry the method:

- `point_id`: a stable ID computed from the address, the chunk's `url`, section `anchor`, and `chunk_num`. Same address, same ID.
- `content_hash`: a fingerprint of the text. Same text, same hash.

In [ ]:
CHUNKS = [
    {"url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
     "anchor": "prerequisites", "chunk_num": 0,
     "text": "Prerequisites - Docker and Docker Compose installed - curl available in your terminal ..."},
    {"url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
     "anchor": "step-2-enable-tls", "chunk_num": 0,
     "text": "Step 2: Enable TLS. Generate a local self-signed certificate and point Qdrant at it ..."},
    {"url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
     "anchor": "step-3-enable-an-admin-api-key", "chunk_num": 0,
     "text": "Step 3: Enable an Admin API Key. Without authentication, anyone with network access ..."},
]

In [ ]:
import hashlib
import uuid

def content_hash(text):
    return hashlib.sha256(text.encode()).hexdigest()

def point_id(url, anchor, num):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"{url}#{anchor}::{num}"))

def prepare(chunks):
    out = []
    for c in chunks:
        # here we can normalize(c["text"]) as recommended in Part 1
        out.append({
            **c,
            "text": c["text"],
            "section_url": f'{c["url"]}#{c["anchor"]}' if c["anchor"] else c["url"],
            "content_hash": content_hash(c["text"]),
            "point_id": point_id(c["url"], c["anchor"], c["chunk_num"]),
        })
    return out

## Buckets: Where Each Chunk Lives

A bucket is one of `N_BUCKETS` slots. A chunk's bucket comes from its `point_id`, the stable ID derived from its address, so editing the text never moves a chunk to a different bucket:

```
point_id = stable ID from (url, anchor, chunk_num)
bucket   = sha256(point_id) mod N_BUCKETS

example (the "Step 2: Enable TLS" chunk):
  point_id = "4e72de03-624c-5b8e-a3ef-93e3a2d3267b"
  bucket   = sha256(point_id) mod 16  =  11
```

The next cell prints the bucket each of our chunks lands in.

In [ ]:
N_BUCKETS = 16
GROUP_SIZE = 4   # buckets packed per group; 16 / 4 = 4 groups

def bucket(pid):
    return int(hashlib.sha256(pid.encode()).hexdigest(), 16) % N_BUCKETS

for c in prepare(CHUNKS):
    print(bucket(c["point_id"]), c["anchor"])

## Digests: One Number per Bucket

Each chunk contributes one number, computed from its ID and its content hash together (the first 64 bits of their hash). A bucket's digest is the XOR of the contributions of its chunks:

```
bucket 5 holds two chunks, each contributes a number (shown tiny, in binary):
  chunk A  ->  0011
  chunk B  ->  0101
  digest   =   0011 XOR 0101  =  0110

edit chunk B's text, so its contribution changes:
  chunk B  ->  1001
  digest   =   0011 XOR 1001  =  1010      (changed: bucket 5 gets investigated)
```

Because the contribution folds in both the ID and the text, any edit, insert, or delete in a bucket changes its digest (a collision is about 2^-64, negligible), and an untouched bucket keeps the exact same digest. That is the signal we compare on.

In [ ]:
def contribution(pid, chash):
    return int(hashlib.sha256((pid + chash).encode()).hexdigest()[:16], 16)  # first 64 bits

def compute_digests(chunks):
    digests = [0] * N_BUCKETS
    for c in chunks:
        digests[bucket(c["point_id"])] ^= contribution(c["point_id"], c["content_hash"])
    return digests

compute_digests(prepare(CHUNKS))

## Storing the Chunks

We store the chunks in a Qdrant collection as in Part 1, with one addition: each point carries its `sync_bucket` in the payload, and we index that field so we can later read a single bucket without scanning the collection.

In [ ]:
MAIN = "docs-sync-scale"
MODEL = "sentence-transformers/all-MiniLM-L6-v2"

client.delete_collection(MAIN)
client.create_collection(
    MAIN,
    vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
)
client.create_payload_index(MAIN, "sync_bucket", models.PayloadSchemaType.INTEGER)

def payload(c):
    return {
        "url": c["url"], "anchor": c["anchor"], "chunk_num": c["chunk_num"],
        "section_url": c["section_url"], "text": c["text"],
        "content_hash": c["content_hash"], "sync_bucket": bucket(c["point_id"]),
    }

def as_points(chunks):
    return [models.PointStruct(id=c["point_id"],
                               vector=models.Document(text=c["text"], model=MODEL),
                               payload=payload(c)) for c in chunks]

client.upsert(MAIN, points=as_points(prepare(CHUNKS)), wait=True)

## The Summary Collection of Digests

We keep the digests in Qdrant too, in a small separate collection. We could store one point per bucket, but at scale that is 65536 points, and reading the whole summary would then mean fetching 65536 points every run, each with Qdrant's per-point overhead. That per-run cost is exactly what we are trying to avoid.

So we pack a group of digests into each point. At scale: 256 points holding 256 digests each (256 x 256 = 65536), so the whole summary is one retrieve of 256 points, and a changed bucket rewrites only the one point its group lives in. A bucket's digest sits at `point = bucket // 256`, `slot = bucket % 256`.

This notebook uses 16 buckets packed 4 per point, so 4 points:

```
  point 0  =  digests for buckets 0, 1, 2, 3
  point 1  =  digests for buckets 4, 5, 6, 7
  point 2  =  digests for buckets 8, 9, 10, 11
  point 3  =  digests for buckets 12, 13, 14, 15

a bucket's digest sits at   point = bucket // 4,   slot = bucket % 4
```

Digests are stored as strings, so their full 64-bit value round-trips without hitting integer limits. The vectors are dummy 1-dimensional values; this collection is never searched.

In [ ]:
META = "docs-sync-digests"
N_META = N_BUCKETS // GROUP_SIZE

client.delete_collection(META)
client.create_collection(
    META,
    vectors_config=models.VectorParams(size=1, distance=models.Distance.COSINE),
)

def write_meta(digests, groups=None):
    """Write digests to the summary collection. Default writes every group."""
    groups = range(N_META) if groups is None else groups
    points = [
        models.PointStruct(
            id=g, vector=[1.0],
            payload={"group": g, "digests": [str(d) for d in digests[g * GROUP_SIZE:(g + 1) * GROUP_SIZE]]})
        for g in groups
    ]
    client.upsert(META, points=points, wait=True)

def read_meta():
    digests = [0] * N_BUCKETS
    for p in client.retrieve(META, ids=list(range(N_META)), with_payload=True):
        g = p.payload["group"]
        for i, d in enumerate(p.payload["digests"]):
            digests[g * GROUP_SIZE + i] = int(d)
    return digests

write_meta(compute_digests(prepare(CHUNKS)))
read_meta()

## Syncing

One run:

1. Compute the digest summary from the current source chunks.
2. Read the stored summary from the summary collection.
3. The buckets where the two differ are the only ones that changed.
4. For each changed bucket: read its chunks from the collection, compare with the source by content hash, embed and upsert what is new or changed, delete what is gone.
5. Rewrite the summary for the changed groups only, after the data writes.

Step 5 runs after the writes on purpose: if a run stops halfway, the summary still points at the unfinished bucket, so the next run redoes it. Redoing is harmless because the writes are keyed by ID and hash.

In [ ]:
def read_bucket(b):
    """The content_hash of every chunk currently stored in bucket b."""
    stored, offset = {}, None
    while True:
        points, offset = client.scroll(
            MAIN,
            scroll_filter=models.Filter(must=[
                models.FieldCondition(key="sync_bucket", match=models.MatchValue(value=b))]),
            with_payload=["content_hash"], with_vectors=False, limit=1000, offset=offset)
        for p in points:
            stored[str(p.id)] = p.payload["content_hash"]
        if offset is None:
            return stored

def reconcile_bucket(b, source_b):
    """Make bucket b in Qdrant match the source. Returns (added, re_embedded, deleted)."""
    stored_b = read_bucket(b)

    added   = [c for pid, c in source_b.items() if pid not in stored_b]
    changed = [c for pid, c in source_b.items()
               if pid in stored_b and stored_b[pid] != c["content_hash"]]
    gone    = [pid for pid in stored_b if pid not in source_b]

    if added or changed:
        client.upsert(MAIN, points=as_points(added + changed), wait=True)
    if gone:
        client.delete(MAIN, points_selector=models.PointIdsList(points=gone), wait=True)
    return len(added), len(changed), len(gone)

def sync(latest_chunks):
    latest = prepare(latest_chunks)

    by_bucket = {}                                    # group the source once, not per changed bucket
    for c in latest:
        by_bucket.setdefault(bucket(c["point_id"]), {})[c["point_id"]] = c

    source = compute_digests(latest)                  # digests from the current source
    stored = read_meta()                              # digests Qdrant holds
    changed_buckets = [b for b in range(N_BUCKETS) if source[b] != stored[b]]

    report = {"changed_buckets": changed_buckets, "added": 0, "re_embedded": 0, "deleted": 0}
    for b in changed_buckets:
        a, r, d = reconcile_bucket(b, by_bucket.get(b, {}))
        report["added"] += a
        report["re_embedded"] += r
        report["deleted"] += d

    changed_groups = {b // GROUP_SIZE for b in changed_buckets}
    write_meta(source, changed_groups)                # rewrite only the changed groups, after the data writes
    return report

## A Month of Edits

We simulate the source after some edits: one chunk unchanged, one text edited, one removed, one new. Running sync should touch only the buckets those changes fall in.

In [ ]:
LATEST = [
    # unchanged
    {"url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
     "anchor": "prerequisites", "chunk_num": 0,
     "text": "Prerequisites - Docker and Docker Compose installed - curl available in your terminal ..."},
    # edited text
    {"url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
     "anchor": "step-2-enable-tls", "chunk_num": 0,
     "text": "Step 2: Enable TLS. Generate a certificate with mkcert and set the TLS config keys ..."},
    # step-3 removed; new step-4 added
    {"url": "https://qdrant.tech/documentation/tutorials-operations/secure-qdrant/",
     "anchor": "step-4-restrict-access", "chunk_num": 0,
     "text": "Step 4: Restrict access with read-only API keys for untrusted clients ..."},
]

sync(LATEST)

The report's `changed_buckets` lists only the buckets holding an edit, insert, or delete, and only the edited and new chunks get re-embedded. The unchanged `prerequisites` chunk sits in another bucket, so it is never fetched.

## Conclusion

The method in one breath: each chunk gets a bucket from its address and a contribution from its text; one small collection holds the XOR digest of each bucket; every run compares the current digests against the stored ones and reconciles only the buckets that differ. The Qdrant reads and re-embeddings then track how much changed, not the size of the whole corpus.

**Use Part 1** when the corpus is small, up to roughly 100k chunks: fewer moving parts, nothing extra to maintain.

**Use this approach** when the corpus is large and each run's changes are small next to its size (around a million chunks and up): a run reads a small summary plus only the changed buckets, instead of the whole collection. Raise the bucket count as the corpus grows, so each bucket stays small.